In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/s5e10-xgb-origcol-20seeds/__results__.html
/kaggle/input/s5e10-xgb-origcol-20seeds/oof_xgb_plus_origcol.csv
/kaggle/input/s5e10-xgb-origcol-20seeds/__notebook__.ipynb
/kaggle/input/s5e10-xgb-origcol-20seeds/__output__.json
/kaggle/input/s5e10-xgb-origcol-20seeds/test_xgb_plus_origcol.csv
/kaggle/input/s5e10-xgb-origcol-20seeds/custom.css
/kaggle/input/s5e10-xgb-origcol-20seeds/__results___files/__results___11_0.png
/kaggle/input/s5e10-lgbm-origcol-20seeds/oof_20seedslgb_plus_origcol.csv
/kaggle/input/s5e10-lgbm-origcol-20seeds/test_20seedslgb_plus_origcol.csv
/kaggle/input/s5e10-lgbm-origcol-20seeds/__results__.html
/kaggle/input/s5e10-lgbm-origcol-20seeds/__notebook__.ipynb
/kaggle/input/s5e10-lgbm-origcol-20seeds/__output__.json
/kaggle/input/s5e10-lgbm-origcol-20seeds/custom.css
/kaggle/input/s5e10-lgbm-origcol-20seeds/__results___files/__results___11_0.png
/kaggle/input/pss5e10-main/lgb_20seed_oof_residuals.csv
/kaggle/input/pss5e10-main/cat_20seed_oof_residuals.csv
/

In [2]:
N_FOLDS=10
SEED=42

In [3]:
oofs_df_baseline = pd.read_csv('/kaggle/input/pss5e10-oofs/oofs_autogluon_baseline.csv')
test_df_baseline = pd.read_csv('/kaggle/input/pss5e10-oofs/test_preds_autogluon_baseline.csv')
oofs_df_residuals = pd.read_csv('/kaggle/input/pss5e10-oofs/oofs_autogluon_residuals.csv')
test_df_residuals = pd.read_csv('/kaggle/input/pss5e10-oofs/test_preds_autogluon_residuals.csv')

oofs_df_residuals.columns = [col + '_res' for col in oofs_df_residuals.columns]
test_df_residuals.columns = [col + '_res' for col in test_df_residuals.columns]

oofs_df_tabm = pd.read_csv('/kaggle/input/s5e10-single-tabm-tuned/oof_tabm_plus_origcol_tuned.csv')
test_df_tabm = pd.read_csv('/kaggle/input/s5e10-single-tabm-tuned/test_tabm_plus_origcol_tuned.csv')

oofs_df_lgbm = pd.read_csv('/kaggle/input/s5e10-lgbm-origcol-20seeds/oof_20seedslgb_plus_origcol.csv')
test_df_lgbm = pd.read_csv('/kaggle/input/s5e10-lgbm-origcol-20seeds/test_20seedslgb_plus_origcol.csv')

oofs_df_xgb = pd.read_csv('/kaggle/input/s5e10-xgb-origcol-20seeds/oof_xgb_plus_origcol.csv')
test_df_xgb = pd.read_csv('/kaggle/input/s5e10-xgb-origcol-20seeds/test_xgb_plus_origcol.csv')

oofs_df_mlp = pd.read_csv('/kaggle/input/s5e10-realmlp-tuned/oof_realmlp_plus_origcol.csv')
test_df_mlp = pd.read_csv('/kaggle/input/s5e10-realmlp-tuned/test_realmlp_plus_origcol.csv')

oofs_df_tabm_res = pd.read_csv('/kaggle/input/s5e10-tabm-over-residuals/oof_tabm_overresid.csv')
test_df_tabm_res = pd.read_csv('/kaggle/input/s5e10-tabm-over-residuals/test_tabm_overresid.csv')

oofs_df_xgb2 = pd.read_csv('/kaggle/input/pss5e10-main/xgb_20seed_oof_residuals.csv')
test_df_xgb2 = pd.read_csv('/kaggle/input/pss5e10-main/xgb_20seed_test_residuals.csv')

oofs_df_cat2 = pd.read_csv('/kaggle/input/pss5e10-main/cat_20seed_oof_residuals.csv')
test_df_cat2 = pd.read_csv('/kaggle/input/pss5e10-main/cat_20seed_test_residuals.csv')

oofs_df_lgb2 = pd.read_csv('/kaggle/input/pss5e10-main/lgb_20seed_oof_residuals.csv')
test_df_lgb2 = pd.read_csv('/kaggle/input/pss5e10-main/lgb_20seed_test_residuals.csv')

oofs_df = pd.concat([
    oofs_df_tabm.drop(columns='id').add_prefix('tabm_'),
    oofs_df_lgbm.drop(columns='id').add_prefix('lgbm_'),
    oofs_df_xgb.drop(columns='id').add_prefix('xgb_'),
    oofs_df_mlp.drop(columns='id').add_prefix('mlp_'),
    oofs_df_tabm_res.drop(columns='id').add_prefix('tabm_res_'),
    # oofs_df_xgb2.drop(columns=['id', 'Unnamed: 0']).add_prefix('xgb2_'),
    oofs_df_cat2.drop(columns=['id', 'Unnamed: 0']).add_prefix('cat2_'),
    oofs_df_lgb2.drop(columns='id').add_prefix('lgb2_')
], axis=1)

test_df = pd.concat([
    test_df_tabm.drop(columns='id').add_prefix('tabm_'),
    test_df_lgbm.drop(columns='id').add_prefix('lgbm_'),
    test_df_xgb.drop(columns='id').add_prefix('xgb_'),
    test_df_mlp.drop(columns='id').add_prefix('mlp_'),
    test_df_tabm_res.drop(columns='id').add_prefix('tabm_res_'),
    # test_df_xgb2.drop(columns=['id', 'Unnamed: 0']).add_prefix('xgb2_'),
    test_df_cat2.drop(columns=['id', 'Unnamed: 0']).add_prefix('cat2_'),
    test_df_lgb2.drop(columns='id').add_prefix('lgb2_')
], axis=1)

# oofs_df = pd.concat([oofs_df_baseline, oofs_df_residuals], axis=1)
# test_df = pd.concat([test_df_baseline, test_df_residuals], axis=1)

train = pd.read_csv('/kaggle/input/playground-series-s5e10/train.csv')
y = train['accident_risk']

In [4]:
oofs_df

,tabm_accident_risk,lgbm_accident_risk,xgb_accident_risk,mlp_accident_risk,tabm_res_accident_risk,cat2_accident_risk,lgb2_accident_risk
0,0.128814,0.128497,0.128963,0.130021,0.130670,0.127302,0.127547
1,0.323260,0.323496,0.322940,0.326364,0.325544,0.324600,0.323805
2,0.382161,0.387007,0.387817,0.383694,0.388800,0.387780,0.389457
3,0.132775,0.129764,0.129615,0.132905,0.134281,0.128532,0.130252
4,0.471251,0.468947,0.469951,0.471781,0.472425,0.471582,0.470061
...,...,...,...,...,...,...,...
517749,0.317979,0.320520,0.320555,0.321173,0.316405,0.318066,0.318041
517750,0.236662,0.237537,0.239382,0.239074,0.240590,0.239583,0.240773
517751,0.296087,0.295426,0.293422,0.294772,0.294203,0.292789,0.291204
517752,0.495484,0.494670,0.493777,0.493962,0.497290,0.494364,0.494947


In [5]:
# # oofs_df.drop(columns='id', inplace=True)
# # test_df.drop(columns='id', inplace=True)

# oofs_df = pd.concat([oofs_df_baseline, oofs_df_residuals, oofs_df], axis=1 )
# test_df = pd.concat([test_df_baseline, test_df_residuals, test_df], axis=1)

In [6]:
# FEATURES = ['WeightedEnsemble_L5', 'WeightedEnsemble_L4', 'LinearModel_BAG_L3',
#        'WeightedEnsemble_L3', 'ExtraTrees_BAG_L2']

# oofs_df = oofs_df[FEATURES]
# test_df = test_df[FEATURES]

In [7]:
def rmse_metric(y_true, y_preds):
    y_true = np.array(y_true)
    y_preds = np.array(y_preds)

    error = y_true - y_preds
    squared = error**2
    squared = np.mean(squared)
    return np.sqrt(squared)

# rmse_metric(y, oofs_df_xgb2['accident_risk'])

for col in oofs_df.columns:
    print('='*15)
    print(f'COLUMN {col} SCORE :', rmse_metric(y, oofs_df[col]))

COLUMN tabm_accident_risk SCORE : 0.05596423208328581
COLUMN lgbm_accident_risk SCORE : 0.055977351985904006
COLUMN xgb_accident_risk SCORE : 0.05596621473299404
COLUMN mlp_accident_risk SCORE : 0.055993027610505564
COLUMN tabm_res_accident_risk SCORE : 0.05593269312329968
COLUMN cat2_accident_risk SCORE : 0.055986679724337435
COLUMN lgb2_accident_risk SCORE : 0.05596857731285173


In [8]:
# ------------------------------------------------------------------
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.metrics import mean_squared_error
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping

# ------------------------------------------------------------------
# 1. Build matrix of meta-features
# ------------------------------------------------------------------
meta_cols = [c for c in oofs_df.columns if c not in {'id','accident_risk'}]
X = oofs_df[meta_cols].copy()
y = train['accident_risk'].copy()
X_test_df = test_df[meta_cols].copy()

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test_df))

SEEDS = [32, 12, 377, 485, 5900, 2392, 3948, 189, 304598, 38950]
# SEEDS = [32, 12, 377, 485]
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# ------------------------------------------------------------------
# 2. Loop over folds
# ------------------------------------------------------------------
# for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
#     print(f'--- Fold {fold+1}/{N_SPLITS} ---')
    
#     X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
#     scaler = StandardScaler()
#     # scaler = QuantileTransformer(output_distribution='normal')
#     X_train_scaled = scaler.fit_transform(X_train)
#     X_val_scaled   = scaler.transform(X_val)
#     X_test_scaled  = scaler.transform(X_test_df)
    
#     # ----------------------------------------------------------
#     # Bagging over seeds inside this fold
#     # ----------------------------------------------------------
#     for seed in SEEDS:
#         np.random.seed(seed)
#         tf.random.set_seed(seed)

#         inp = Input(shape=(X_train_scaled.shape[1],))
#         x = Dense(32, activation='relu')(inp)
#         # gate = Dense(64, activation='sigmoid')(inp)
#         # x = tf.keras.layers.multiply([x, gate])
#         x = Dense(16, activation='relu')(x)
#         out = Dense(1)(x)

#         model = Model(inp, out)
        
#         # model = Sequential([
#         #     Input(shape=(X_train_scaled.shape[1],)),
#         #     Dense(64, activation='relu'),
#         #     Dense(32, activation='relu'),
#         #     Dense(1)
#         # ])
#         model.compile(optimizer='adam', loss='mean_squared_error')
        
#         es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
        
#         model.fit(X_train_scaled, y_train,
#                   validation_data=(X_val_scaled, y_val),
#                   epochs=200, batch_size=512,
#                   callbacks=[es], verbose=0)
        
#         val_pred = model.predict(X_val_scaled, verbose=0).ravel()
#         oof_preds[val_idx] += val_pred / len(SEEDS)
        
#         test_preds += model.predict(X_test_scaled, verbose=0).ravel() / len(SEEDS)
    
#     fold_rmse = mean_squared_error(y_val, oof_preds[val_idx], squared=False)
#     print(f"Fold {fold+1} RMSE: {fold_rmse:.5f}")

# # ------------------------------------------------------------------
# # 3. Average test predictions across folds & report overall RMSE
# # ------------------------------------------------------------------
# test_preds /= N_SPLITS
# overall_oof_rmse = mean_squared_error(y, oof_preds, squared=False)
# print(f"\nOverall OOF RMSE: {overall_oof_rmse:.5f}")

2025-10-18 17:43:46.182470: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760809426.549083      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760809426.636496      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [9]:
# ------------------------------------------------------------------
#  FIXED meta-learner  (no tfa, no schedule conflict)
# ------------------------------------------------------------------
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.isotonic import IsotonicRegression
from tensorflow.keras import layers as L
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

# ---------- config -----------------------------------------------------------
N_META      = X.shape[1]
N_SPLITS    = 5
N_SEEDS     = len(SEEDS)
N_MC        = 25
META_UNITS  = 64
N_EXPERTS   = 3

tf.keras.utils.set_random_seed(42)

# ---------- model ------------------------------------------------------------
def build_meta_model():
    inp = L.Input(shape=(N_META,))
    raw  = inp
    rank = L.Lambda(lambda t: tf.cast(tf.argsort(tf.argsort(t, axis=1), axis=1), tf.float32)
                          / tf.cast(tf.shape(t)[1], tf.float32))(inp)
    h    = L.Concatenate()([raw, rank])

    gates = L.Dense(N_EXPERTS, activation='softmax')(h)
    experts = []
    for _ in range(N_EXPERTS):
        x = L.Dense(META_UNITS, activation='swish')(h)
        x = L.Dropout(0.15)(x)
        x = L.Dense(META_UNITS // 2, activation='swish')(x)
        experts.append(L.Dense(1)(x))
    stack = L.Concatenate()(experts)
    out   = L.Dot(axes=1)([stack, gates])
    return Model(inp, out)

# ---------- CV loop ----------------------------------------------------------
oof_preds_new = np.zeros(len(X))
test_preds_new = np.zeros(len(X_test_df))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f'--- Meta fold {fold+1}/{N_SPLITS} ---')
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train)
    X_va = scaler.transform(X_val)
    X_te = scaler.transform(X_test_df)

    fold_val = np.zeros_like(y_val, dtype=np.float32)
    fold_tst = np.zeros(len(X_te), dtype=np.float32)

    for seed in SEEDS:
        tf.keras.utils.set_random_seed(seed)
        model = build_meta_model()
        # plain float LR -> ReduceLROnPlateau can touch it
        opt = tf.keras.optimizers.Adam(learning_rate=3e-3, clipnorm=1.0)
        model.compile(optimizer=opt, loss='mse')

        cbs = [
            EarlyStopping(patience=30, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(patience=10, factor=0.3)
        ]

        model.fit(X_tr, y_train,
                  validation_data=(X_va, y_val),
                  epochs=300,
                  batch_size=512,
                  callbacks=cbs,
                  verbose=0)

        # MC-dropout
        val_mc = np.stack([model(X_va, training=True).numpy().ravel()
                           for _ in range(N_MC)]).mean(axis=0)
        tst_mc = np.stack([model(X_te, training=True).numpy().ravel()
                           for _ in range(N_MC)]).mean(axis=0)

        fold_val += val_mc / N_SEEDS
        fold_tst += tst_mc / N_SEEDS

    oof_preds_new[val_idx] = fold_val
    test_preds_new += fold_tst / N_SPLITS
    print(f'Fold {fold+1} RMSE: {mean_squared_error(y_val, fold_val):.5f}')

# ---------- calibration ------------------------------------------------------
iso = IsotonicRegression(out_of_bounds='clip')
oof_cal = iso.fit_transform(oof_preds_new, y)
test_cal = iso.transform(test_preds_new)

print(f'\nIsotonic-calibrated OOF RMSE: {mean_squared_error(y, oof_cal, squared=False):.5f}')

pd.DataFrame({'id': range(len(train)), 'accident_risk': oof_cal}) \
  .to_csv('meta_moe_mc_isotonic_oof.csv', index=False)
pd.DataFrame({'id': range(len(test_df)),  'accident_risk': test_cal}) \
  .to_csv('meta_moe_mc_isotonic_test.csv', index=False)

--- Meta fold 1/5 ---


2025-10-18 17:44:04.936640: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Fold 1 RMSE: 0.00314
--- Meta fold 2/5 ---
Fold 2 RMSE: 0.00312
--- Meta fold 3/5 ---
Fold 3 RMSE: 0.00313
--- Meta fold 4/5 ---
Fold 4 RMSE: 0.00311
--- Meta fold 5/5 ---
Fold 5 RMSE: 0.00311

Isotonic-calibrated OOF RMSE: 0.05583


In [10]:
# # ------------------------------------------------------------------
# #  ONE-SHOT  multi-meta-model tester  (expanded & cleaned)
# # ------------------------------------------------------------------
# import numpy as np, pandas as pd, tensorflow as tf
# from sklearn.model_selection import KFold
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics import mean_squared_error
# from sklearn.isotonic import IsotonicRegression
# from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
# from sklearn.linear_model import BayesianRidge, HuberRegressor, PassiveAggressiveRegressor
# from sklearn.neighbors import KNeighborsRegressor
# from sklearn.svm import SVR
# from tensorflow.keras.layers import Dense, Input, Dropout
# from tensorflow.keras.models import Model
# from tensorflow.keras.callbacks import EarlyStopping
# import warnings
# warnings.filterwarnings('ignore')

# # ---------- data -------------------------------------------------------------
# meta_cols = [c for c in oofs_df.columns if c not in {'id','accident_risk'}]
# X = oofs_df[meta_cols].copy()
# y = train['accident_risk'].copy()
# X_test_df = test_df[meta_cols].copy()

# SEEDS = [32, 12, 377, 485, 5900]
# N_SPLITS = 5
# kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# # ---------- choose models ----------------------------------------------------
# MODELS = {
#     # 'nn_simple' : True,   # your original 32-16-1 relu
#     # 'et'        : True,   # Extra-Trees
#     'gbm'       : True,   # Sklearn GBM
#     # 'rf'        : True,   # Random-Forest
#     'br'        : True,   # Bayesian Ridge (linear but probabilistic)
#     'huber'     : True,   # Huber regressor (robust)
#     'par'       : True,   # Passive-Aggressive (large-margin)
#     'knn'       : True,   # k-NN regression
#     'svr'       : True,   # linear SVR (fast, probability-friendly)
# }

# # ---------- NN helper --------------------------------------------------------
# def nn_simple_block(X_tr, y_tr, X_va, X_te, seed):
#     tf.random.set_seed(seed)
#     inp = Input(shape=(X_tr.shape[1],))
#     x = Dense(32, activation='relu')(inp)
#     x = Dense(16, activation='relu')(x)
#     out = Dense(1)(x)
#     model = Model(inp, out)
#     model.compile('adam', 'mse')
#     es = EarlyStopping(patience=20, restore_best_weights=True)
#     model.fit(X_tr, y_tr, validation_data=(X_va, y_tr),
#               epochs=200, batch_size=512, callbacks=[es], verbose=0)
#     return model.predict(X_va, verbose=0).ravel(), model.predict(X_te, verbose=0).ravel()

# # ---------- generic sklearn helper ------------------------------------------
# def sklearn_block(clf, X_tr, y_tr, X_va, X_te):
#     clf.fit(X_tr, y_tr)
#     return clf.predict(X_va), clf.predict(X_te)

# # ---------- CV loop ---------------------------------------------------------
# results = {}
# scaler = StandardScaler()

# for mname, run in MODELS.items():
#     if not run: continue
#     print(f'\n=====  {mname.upper()}  =====')
#     oof = np.zeros(len(X))
#     tst = np.zeros(len(X_test_df))

#     for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
#         print(f'Fold {fold+1}/{N_SPLITS}', end=' | ')
#         X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#         y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

#         X_tr = scaler.fit_transform(X_train)
#         X_va = scaler.transform(X_val)
#         X_te = scaler.transform(X_test_df)

#         fold_val = np.zeros_like(y_val, dtype=np.float32)
#         fold_tst = np.zeros(len(X_te), dtype=np.float32)

#         for seed in SEEDS:
#             # ---------- model factory ---------------------------------
#             if mname == 'nn_simple':
#                 v, t = nn_simple_block(X_tr, y_train, X_va, X_te, seed)
#             elif mname == 'et':
#                 clf = ExtraTreesRegressor(n_estimators=1000, max_depth=None,
#                                           min_samples_split=5, max_features=0.6,
#                                           bootstrap=True, random_state=seed, n_jobs=-1)
#                 v, t = sklearn_block(clf, X_tr, y_train, X_va, X_te)
#             elif mname == 'gbm':
#                 clf = GradientBoostingRegressor(n_estimators=1500, max_depth=4,
#                                                 learning_rate=0.02, subsample=0.8,
#                                                 random_state=seed)
#                 v, t = sklearn_block(clf, X_tr, y_train, X_va, X_te)
#             elif mname == 'rf':
#                 clf = RandomForestRegressor(n_estimators=1000, max_depth=None,
#                                             min_samples_split=5, max_features=0.6,
#                                             random_state=seed, n_jobs=-1)
#                 v, t = sklearn_block(clf, X_tr, y_train, X_va, X_te)
#             elif mname == 'br':
#                 clf = BayesianRidge(alpha_1=1e-6, alpha_2=1e-6, lambda_1=1e-6, lambda_2=1e-6)
#                 v, t = sklearn_block(clf, X_tr, y_train, X_va, X_te)
#             elif mname == 'huber':
#                 clf = HuberRegressor(epsilon=1.35, alpha=0.001, max_iter=1000)
#                 v, t = sklearn_block(clf, X_tr, y_train, X_va, X_te)
#             elif mname == 'par':
#                 clf = PassiveAggressiveRegressor(C=1.0, epsilon=0.01, max_iter=1000, random_state=seed)
#                 v, t = sklearn_block(clf, X_tr, y_train, X_va, X_te)
#             elif mname == 'knn':
#                 clf = KNeighborsRegressor(n_neighbors=15, weights='distance', metric='euclidean', n_jobs=-1)
#                 v, t = sklearn_block(clf, X_tr, y_train, X_va, X_te)
#             elif mname == 'svr':
#                 clf = SVR(kernel='linear', C=1.0, epsilon=0.01)
#                 v, t = sklearn_block(clf, X_tr, y_train, X_va, X_te)
#             else:
#                 raise ValueError(mname)

#             fold_val += v / len(SEEDS)
#             fold_tst += t / len(SEEDS)

#         oof[val_idx] = fold_val
#         tst += fold_tst / N_SPLITS
#         print(f'RMSE: {mean_squared_error(y_val, fold_val):.5f}')

#     # ---- isotonic calibration ----
#     iso = IsotonicRegression(out_of_bounds='clip')
#     oof_cal = iso.fit_transform(oof, y)
#     tst_cal = iso.transform(tst)
#     final_rmse = mean_squared_error(y, oof_cal, squared=False)
#     print(f'{mname}  -->  calibrated OOF RMSE: {final_rmse:.5f}')

#     # ---- save ----
#     results[mname] = final_rmse
#     pd.DataFrame({'id': range(len(train)), 'accident_risk': oof_cal}) \
#       .to_csv(f'{mname}_oof.csv', index=False)
#     pd.DataFrame({'id': range(len(test_df)),  'accident_risk': tst_cal}) \
#       .to_csv(f'{mname}_test.csv', index=False)

# # ---------- quick summary ----------------------------------------------------
# print('\n===  SUMMARY  ===')
# for k, v in results.items():
#     print(f'{k:8s} : {v:.5f}')

In [11]:
# samp = pd.read_csv('/kaggle/input/playground-series-s5e10/sample_submission.csv')
# samp['accident_risk'] = test_preds
# samp.to_csv('submission_nn_meta_7models.csv', index=False)

In [12]:
# from sklearn.model_selection import KFold
# from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge, HuberRegressor, LinearRegression
# from sklearn.kernel_approximation import RBFSampler
# from sklearn.pipeline import make_pipeline
# from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
# from sklearn.neural_network import MLPRegressor
# from sklearn.svm import SVR
# from sklearn.preprocessing import StandardScaler
# # from sklearn
# from sklearn.base import clone
    
    
# def cv_score(model_dict, X, y, N_FOLDS=N_FOLDS, SEED=SEED):
#     kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
#     for name, model in model_dict.items():
#         print('='*20)
#         print(f'PROCESSING MODEL :{name}')
#         print('='*20)
#         scores = []
#         for i, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
#             reg = clone(model)
#             X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#             y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

#             reg.fit(X_train, y_train)
#             preds = reg.predict(X_val)
#             score = rmse(y_val, preds)
#             scores.append(score)
#             print(f'SCORE FOR FOLD{i}: {score}')
#         print(f'MEAN SCORE ACROSS ALL FOLDS: {np.mean(scores)}')

# model_dict = {
#     'lr': LinearRegression(n_jobs=-1),
#     # 'kr': KernelRidge(alpha=0.001),
#     # 'kr1': KernelRidge(alpha=0.01),
#     # 'kr2': KernelRidge(alpha=0.1),
#     # 'etr': ExtraTreesRegressor(n_jobs=-1)
# }
# meta_models = {
#     # ----  linear / ridge / lasso  ----
#     'ridge': Ridge(alpha=0.0001, max_iter=100000),
#     # 'lasso': Lasso(alpha=0.001, max_iter=10_000),
#     # # 'enet':  ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10_00),
#     # 'bayes': BayesianRidge(),
#     # 'huber': HuberRegressor(epsilon=1.35, max_iter=10_000),

#     # ----  tree ensembles  ----
#     # 'et':    ExtraTreesRegressor(n_estimators=300, max_depth=8, n_jobs=-1, random_state=SEED),
#     # 'rf':    RandomForestRegressor(n_estimators=300, max_depth=8, n_jobs=-1, random_state=SEED),
#     # 'gb':    GradientBoostingRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
#                                        # subsample=0.8, random_state=SEED),

#     # ----  small neural net  ----
# #     'mlp':   make_pipeline(StandardScaler(),
# #                            MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500,
# #                                         early_stopping=True, random_state=SEED)),

# #     # ----  kernel approx + ridge  ----
# #     'rbf_ridge': make_pipeline(StandardScaler(),
# #                                RBFSampler(gamma=0.5, n_components=500, random_state=SEED),
# #                                Ridge(alpha=0.1)),

# #     # ----  linear SVR  ----
# #     'svr': make_pipeline(StandardScaler(),
# #                         SVR(kernel='linear', C=1.0, epsilon=0.01)),
#  }

# cv_score(meta_models, oofs_df, y)

In [13]:
# meta = Ridge(alpha=0.001, max_iter=10000)
# meta.fit(oofs_df, y)

In [14]:
# import numpy as np
# import pandas as pd
# from scipy.optimize import minimize, differential_evolution
# from sklearn.metrics import mean_squared_error
# from sklearn.cluster import KMeans
# from sklearn.metrics import silhouette_score
# import warnings
# warnings.filterwarnings('ignore')

# def detailed_ensemble_comparison(oofs_df, test_df, y_true, SEED=42):
#     """
#     Comprehensive ensemble comparison with detailed logging
#     """
#     print("=" * 80)
#     print("ENSEMBLE METHOD COMPARISON")
#     print("=" * 80)
#     print(f"Input shapes - OOF: {oofs_df.shape}, Test: {test_df.shape}, Target: {y_true.shape}")
#     print(f"Number of base models: {oofs_df.shape[1]}")
#     print(f"Target range: {y_true.min():.4f} to {y_true.max():.4f}")
#     print()
    
#     # Convert to numpy arrays for faster computation
#     oof_preds = oofs_df.values
#     test_preds = test_df.values
    
#     results = {}
    
#     # Method 1: Simple Average
#     print("1. SIMPLE AVERAGE")
#     print("-" * 40)
#     oof_avg = np.mean(oof_preds, axis=1)
#     test_avg = np.mean(test_preds, axis=1)
#     rmse_avg = mean_squared_error(y_true, oof_avg, squared=False)
#     results['simple_avg'] = {'oof': oof_avg, 'test': test_avg, 'rmse': rmse_avg}
#     print(f"   RMSE: {rmse_avg:.6f}")
#     print(f"   OOF range: {oof_avg.min():.6f} to {oof_avg.max():.6f}")
#     print()
    
#     # Method 2: Median
#     print("2. MEDIAN")
#     print("-" * 40)
#     oof_median = np.median(oof_preds, axis=1)
#     test_median = np.median(test_preds, axis=1)
#     rmse_median = mean_squared_error(y_true, oof_median, squared=False)
#     results['median'] = {'oof': oof_median, 'test': test_median, 'rmse': rmse_median}
#     print(f"   RMSE: {rmse_median:.6f}")
#     print(f"   OOF range: {oof_median.min():.6f} to {oof_median.max():.6f}")
#     print()
    
#     # Method 3: Best Single Model
#     print("3. BEST SINGLE MODEL")
#     print("-" * 40)
#     single_scores = []
#     for i in range(oof_preds.shape[1]):
#         rmse = mean_squared_error(y_true, oof_preds[:, i], squared=False)
#         single_scores.append(rmse)
    
#     best_idx = np.argmin(single_scores)
#     best_rmse = single_scores[best_idx]
#     results['best_single'] = {
#         'oof': oof_preds[:, best_idx], 
#         'test': test_preds[:, best_idx], 
#         'rmse': best_rmse,
#         'model_idx': best_idx
#     }
#     print(f"   Best model index: {best_idx}")
#     print(f"   Best single model RMSE: {best_rmse:.6f}")
#     print(f"   All single model RMSE range: {min(single_scores):.6f} to {max(single_scores):.6f}")
#     print(f"   Std of single model RMSE: {np.std(single_scores):.6f}")
#     print()
    
#     # Method 4: Top-K Average
#     print("4. TOP-K MODELS AVERAGE")
#     print("-" * 40)
#     k_values = [3, 5, 10]
#     for k in k_values:
#         top_k_idx = np.argsort(single_scores)[:k]
#         oof_topk = np.mean(oof_preds[:, top_k_idx], axis=1)
#         test_topk = np.mean(test_preds[:, top_k_idx], axis=1)
#         rmse_topk = mean_squared_error(y_true, oof_topk, squared=False)
#         results[f'top{k}_avg'] = {'oof': oof_topk, 'test': test_topk, 'rmse': rmse_topk}
#         print(f"   Top-{k}: RMSE = {rmse_topk:.6f} (models: {top_k_idx})")
#     print()
    
#     # Method 5: Performance-Weighted Average
#     print("5. PERFORMANCE-WEIGHTED AVERAGE")
#     print("-" * 40)
#     # Use inverse RMSE as weights (better models get higher weight)
#     weights = 1 / (np.array(single_scores) + 1e-8)
#     weights = weights / np.sum(weights)
    
#     oof_weighted = np.average(oof_preds, axis=1, weights=weights)
#     test_weighted = np.average(test_preds, axis=1, weights=weights)
#     rmse_weighted = mean_squared_error(y_true, oof_weighted, squared=False)
#     results['weighted_avg'] = {'oof': oof_weighted, 'test': test_weighted, 'rmse': rmse_weighted}
#     print(f"   RMSE: {rmse_weighted:.6f}")
#     print(f"   Weight range: {weights.min():.6f} to {weights.max():.6f}")
#     print(f"   Top 3 models by weight: {np.argsort(weights)[-3:][::-1]}")
#     print()
    
#     # Method 6: Trimmed Mean
#     print("6. TRIMMED MEAN ENSEMBLE")
#     print("-" * 40)
#     trim_percentages = [5, 10, 15]
#     for trim_pct in trim_percentages:
#         trim_count = int(oof_preds.shape[1] * trim_pct / 100)
#         oof_sorted = np.sort(oof_preds, axis=1)
#         test_sorted = np.sort(test_preds, axis=1)
        
#         if trim_count > 0:
#             oof_trimmed = oof_sorted[:, trim_count:-trim_count]
#             test_trimmed = test_sorted[:, trim_count:-trim_count]
#         else:
#             oof_trimmed = oof_sorted
#             test_trimmed = test_sorted
        
#         oof_trim = np.mean(oof_trimmed, axis=1)
#         test_trim = np.mean(test_trimmed, axis=1)
#         rmse_trim = mean_squared_error(y_true, oof_trim, squared=False)
#         results[f'trimmed_{trim_pct}pct'] = {'oof': oof_trim, 'test': test_trim, 'rmse': rmse_trim}
#         print(f"   Trimmed {trim_pct}%: RMSE = {rmse_trim:.6f} (using {oof_trimmed.shape[1]} models)")
#     print()
    
#     # Method 7: Winsorized Mean
#     print("7. WINSORIZED MEAN")
#     print("-" * 40)
#     win_percentages = [5, 10]
#     for win_pct in win_percentages:
#         lower = np.percentile(oof_preds, win_pct, axis=1)
#         upper = np.percentile(oof_preds, 100 - win_pct, axis=1)
        
#         oof_winsor = np.array([np.mean(np.clip(row, lower[i], upper[i])) 
#                               for i, row in enumerate(oof_preds)])
        
#         test_lower = np.percentile(test_preds, win_pct, axis=1)
#         test_upper = np.percentile(test_preds, 100 - win_pct, axis=1)
#         test_winsor = np.array([np.mean(np.clip(row, test_lower[i], test_upper[i])) 
#                                for i, row in enumerate(test_preds)])
        
#         rmse_winsor = mean_squared_error(y_true, oof_winsor, squared=False)
#         results[f'winsorized_{win_pct}pct'] = {'oof': oof_winsor, 'test': test_winsor, 'rmse': rmse_winsor}
#         print(f"   Winsorized {win_pct}%: RMSE = {rmse_winsor:.6f}")
#     print()
    
#     # Method 8: Correlation-Based Ensemble Selection
#     print("8. CORRELATION-BASED ENSEMBLE SELECTION")
#     print("-" * 40)
#     try:
#         corr_matrix = np.corrcoef(oof_preds.T)
#         print(f"   Average correlation between models: {np.mean(corr_matrix):.4f}")
        
#         max_correlations = [0.90, 0.95, 0.98]
#         for max_corr in max_correlations:
#             selected_models = [0]
#             for i in range(1, oof_preds.shape[1]):
#                 max_corr_with_selected = max([corr_matrix[i, j] for j in selected_models])
#                 if max_corr_with_selected < max_corr:
#                     selected_models.append(i)
            
#             oof_corr = oof_preds[:, selected_models]
#             test_corr = test_preds[:, selected_models]
            
#             # Weight by performance
#             corr_scores = [1 / (single_scores[i] + 1e-8) for i in selected_models]
#             corr_weights = np.array(corr_scores) / np.sum(corr_scores)
            
#             oof_final = np.average(oof_corr, axis=1, weights=corr_weights)
#             test_final = np.average(test_corr, axis=1, weights=corr_weights)
#             rmse_corr = mean_squared_error(y_true, oof_final, squared=False)
            
#             results[f'corr_based_{int(max_corr*100)}'] = {
#                 'oof': oof_final, 'test': test_final, 'rmse': rmse_corr,
#                 'selected_models': selected_models
#             }
#             print(f"   Max correlation {max_corr}: {len(selected_models)} models, RMSE = {rmse_corr:.6f}")
#     except Exception as e:
#         print(f"   Correlation-based method failed: {e}")
#     print()
    
#     # Method 9: Clustering-Based Ensemble
#     print("9. CLUSTERING-BASED ENSEMBLE")
#     print("-" * 40)
#     try:
#         # Find optimal number of clusters
#         best_score = -1
#         best_n_clusters = 2
        
#         for n_clusters in range(2, min(8, oof_preds.shape[1]//2)):
#             kmeans = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10)
#             labels = kmeans.fit_predict(oof_preds.T)
#             score = silhouette_score(oof_preds.T, labels)
#             if score > best_score:
#                 best_score = score
#                 best_n_clusters = n_clusters
        
#         print(f"   Optimal clusters: {best_n_clusters} (silhouette: {best_score:.4f})")
        
#         kmeans = KMeans(n_clusters=best_n_clusters, random_state=SEED, n_init=10)
#         cluster_labels = kmeans.fit_predict(oof_preds.T)
        
#         # Select best model from each cluster
#         selected_models = []
#         for cluster_id in range(best_n_clusters):
#             cluster_models = np.where(cluster_labels == cluster_id)[0]
#             best_model = min(cluster_models, key=lambda x: single_scores[x])
#             selected_models.append(best_model)
        
#         # Ensemble selected models
#         oof_cluster = oof_preds[:, selected_models]
#         test_cluster = test_preds[:, selected_models]
        
#         cluster_scores = [1 / (single_scores[i] + 1e-8) for i in selected_models]
#         cluster_weights = np.array(cluster_scores) / np.sum(cluster_scores)
        
#         oof_final = np.average(oof_cluster, axis=1, weights=cluster_weights)
#         test_final = np.average(test_cluster, axis=1, weights=cluster_weights)
#         rmse_cluster = mean_squared_error(y_true, oof_final, squared=False)
        
#         results['clustering'] = {
#             'oof': oof_final, 'test': test_final, 'rmse': rmse_cluster,
#             'selected_models': selected_models, 'n_clusters': best_n_clusters
#         }
#         print(f"   RMSE: {rmse_cluster:.6f}")
#         print(f"   Selected models from clusters: {selected_models}")
#     except Exception as e:
#         print(f"   Clustering method failed: {e}")
#     print()
    
#     # Method 10: Bayesian Optimization Weights
#     print("10. BAYESIAN OPTIMIZATION WEIGHTS")
#     print("-" * 40)
#     try:
#         def objective(weights):
#             weights = np.abs(weights)
#             weights = weights / np.sum(weights)
#             blended = np.average(oof_preds, axis=1, weights=weights)
#             return mean_squared_error(y_true, blended, squared=False)
        
#         bounds = [(0, 1) for _ in range(oof_preds.shape[1])]
#         result = differential_evolution(objective, bounds, maxiter=100, popsize=15, seed=SEED)
        
#         optimal_weights = np.abs(result.x) / np.sum(np.abs(result.x))
        
#         oof_bayesian = np.average(oof_preds, axis=1, weights=optimal_weights)
#         test_bayesian = np.average(test_preds, axis=1, weights=optimal_weights)
#         rmse_bayesian = mean_squared_error(y_true, oof_bayesian, squared=False)
        
#         results['bayesian_opt'] = {
#             'oof': oof_bayesian, 'test': test_bayesian, 'rmse': rmse_bayesian,
#             'weights': optimal_weights
#         }
#         print(f"   RMSE: {rmse_bayesian:.6f}")
#         print(f"   Optimization success: {result.success}")
#         print(f"   Weight range: {optimal_weights.min():.6f} to {optimal_weights.max():.6f}")
#     except Exception as e:
#         print(f"   Bayesian optimization failed: {e}")
#     print()
    
#     # Method 11: Geometric Mean (for probabilities)
#     print("11. GEOMETRIC MEAN")
#     print("-" * 40)
#     try:
#         # Add small epsilon to avoid log(0)
#         epsilon = 1e-8
#         oof_geo = np.exp(np.mean(np.log(np.clip(oof_preds, epsilon, 1-epsilon)), axis=1))
#         test_geo = np.exp(np.mean(np.log(np.clip(test_preds, epsilon, 1-epsilon)), axis=1))
#         rmse_geo = mean_squared_error(y_true, oof_geo, squared=False)
#         results['geometric_mean'] = {'oof': oof_geo, 'test': test_geo, 'rmse': rmse_geo}
#         print(f"   RMSE: {rmse_geo:.6f}")
#     except Exception as e:
#         print(f"   Geometric mean failed: {e}")
#     print()
    
#     # Final Results Summary
#     print("=" * 80)
#     print("FINAL RESULTS SUMMARY")
#     print("=" * 80)
    
#     # Sort methods by RMSE
#     sorted_results = sorted(results.items(), key=lambda x: x[1]['rmse'])
    
#     print("\nRanked by RMSE (lower is better):")
#     print("-" * 60)
#     for i, (method, data) in enumerate(sorted_results, 1):
#         improvement = ((results['simple_avg']['rmse'] - data['rmse']) / results['simple_avg']['rmse']) * 100
#         print(f"{i:2d}. {method:20} RMSE: {data['rmse']:.6f} "
#               f"({improvement:+.2f}% vs simple avg)")
    
#     # Best method
#     best_method, best_data = sorted_results[0]
#     print(f"\n🎯 BEST METHOD: {best_method} with RMSE: {best_data['rmse']:.6f}")
    
#     return results, best_method

# # Run the comprehensive comparison
# print("Starting ensemble comparison...")
# results, best_method = detailed_ensemble_comparison(oofs_df, test_df, y.values, SEED=42)

# # Get best predictions
# best_oof = results[best_method]['oof']
# best_test = results[best_method]['test']

# print(f"\nBest OOF predictions shape: {best_oof.shape}")
# print(f"Best test predictions shape: {best_test.shape}")
# print(f"Best OOF range: {best_oof.min():.6f} to {best_oof.max():.6f}")

In [15]:
# # ==========================================
# #  DATA-DRIVEN  h-blend  (CV only)  S5E10
# # ==========================================
# import numpy as np, pandas as pd
# from tqdm.auto import tqdm
# from sklearn.metrics import mean_squared_error

# # ---------- 1.  rank-transform ----------
# def rank_transform(df):
#     return df.rank(pct=True, axis=0)

# oof_rank  = rank_transform(oofs_df)
# test_rank = rank_transform(test_df)

# # ---------- 2.  pick 10 best single models ----------
# cv_scores = np.array([mean_squared_error(y, oofs_df.iloc[:,i], squared=False)
#                       for i in range(oofs_df.shape[1])])
# top10_idx = np.argsort(cv_scores)[:10]
# top10_cols = oofs_df.columns[top10_idx].tolist()
# print('Top-10 models (CV):', [f'{c} ({cv_scores[i]:.6f})' for c, i in zip(top10_cols, top10_idx)])

# # ---------- 3.  pair builder ----------
# pairs = [(top10_cols[i], top10_cols[i+1]) for i in range(0, 10, 2)]

# # ---------- 4.  CV tuner ----------
# def tune_pair(oof_a, oof_b, y, steps=21):
#     best_rmse = np.inf
#     for w in np.linspace(0.25, 0.75, steps):
#         pred = w*oof_a + (1-w)*oof_b
#         rmse = mean_squared_error(y, pred, squared=False)
#         if rmse < best_rmse:
#             best_rmse, best_w = rmse, w
#     return best_w, 1-best_w, best_rmse

# group_oof, group_test, group_w = [], [], []
# for c1, c2 in tqdm(pairs, desc='Tuning pairs'):
#     w1, w2, rmse = tune_pair(oof_rank[c1], oof_rank[c2], y)
#     group_w.append([w1, w2])
#     group_oof.append(w1*oof_rank[c1] + w2*oof_rank[c2])
#     group_test.append(w1*test_rank[c1] + w2*test_rank[c2])

# # ---------- 5.  second-level CV tuner ----------
# group_oof_df  = pd.DataFrame(np.column_stack(group_oof), columns=[f'g{i}' for i in range(1,6)])
# group_test_df = pd.DataFrame(np.column_stack(group_test), columns=[f'g{i}' for i in range(1,6)])

# best_rmse, best_w = np.inf, None
# for w1 in tqdm(np.linspace(0.02, 0.80, 40), desc='Tuning 5-group weights'):
#     for w2 in np.linspace(0.02, 0.80, 40):
#         if w1+w2 >= 0.98: continue
#         w3 = (1-w1-w2)/3
#         w4, w5 = w3, w3
#         w = np.array([w1, w2, w3, w4, w5])
#         pred = group_oof_df.dot(w)
#         rmse = mean_squared_error(y, pred, squared=False)
#         if rmse < best_rmse:
#             best_rmse, best_w = rmse, w

# print('Best 5-group weights (CV):', best_w.round(4), 'RMSE:', best_rmse)

# # ---------- 6.  final blend ----------
# oof_final  = group_oof_df.dot(best_w)
# test_final = group_test_df.dot(best_w)
# oof_final  = np.clip(oof_final, 0, 1)
# test_final = np.clip(test_final, 0, 1)

# # ---------- 7.  submission ----------
# sub = pd.read_csv('/kaggle/input/playground-series-s5e10/sample_submission.csv')
# sub['accident_risk'] = test_final
# sub.to_csv('h_blend_cv_tuned.csv', index=False)
# print('Saved → h_blend_cv_tuned.csv')

In [16]:
# from sklearn.isotonic import IsotonicRegression

# # 1. calibration object fitted on TRUTH vs rank-blend
# ir = IsotonicRegression(out_of_bounds='clip')
# ir.fit(oof_final, y)          # X = quantile, y = true accident_risk

# # 2. map both OOF and test back to original scale
# oof_final_calib  = ir.transform(oof_final)
# test_final_calib = ir.transform(test_final)

# print('Calibrated CV RMSE:', mean_squared_error(y, oof_final_calib, squared=False))